# Core Order-Level Analysis Dataset

## Purpose

This notebook constructs the core order-level dataset used in the statistical analysis and Power BI dashboard.

Each row in the final dataset represents one order. The dataset will combine delivery information with customer-review scores and create the main analytical variables.

## Main tasks

- load the orders and reviews tables;
- retain delivered orders with the required delivery dates;
- handle orders with multiple review records;
- merge delivery information with customer-review scores;
- calculate delivery delay days;
- classify orders as delayed or on time;
- validate that each row represents one unique order;
- export the processed dataset for further analysis and visualization.

## Unit of analysis

The unit of analysis is one order. Therefore, `order_id` must be unique in the final dataset.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
orders_path = raw_data_dir / "olist_orders_dataset.csv"
reviews_path = raw_data_dir / "olist_order_reviews_dataset.csv"

print("Project root:", project_root.resolve())
print("Orders file exists:", orders_path.exists())
print("Reviews file exists:", reviews_path.exists())

Project root: F:\Personal_interesting_projects\ecommerce-delivery-analysis\Main file\ecommerce-delivery-analysis
Orders file exists: True
Reviews file exists: True


## 1. Load the Core Tables

The orders table contains delivery information, while the reviews table contains customer-review scores.

In [2]:
orders = pd.read_csv(
    orders_path,
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

reviews = pd.read_csv(
    reviews_path,
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

print("Orders table shape:", orders.shape)
print("Reviews table shape:", reviews.shape)

display(orders.head(3))
display(reviews.head(3))

Orders table shape: (99441, 8)
Reviews table shape: (99224, 7)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24


## 2. Check the Unit of Analysis

Before merging the tables, we verify whether each order appears only once in the orders and reviews tables. Multiple review records for one order must be handled to preserve one row per order.

In [3]:
duplicated_order_rows = orders["order_id"].duplicated().sum()
duplicated_review_id_rows = reviews["review_id"].duplicated().sum()
exact_duplicate_review_rows = reviews.duplicated().sum()

review_counts_per_order = reviews.groupby("order_id").size()
orders_with_multiple_reviews = (review_counts_per_order > 1).sum()
extra_review_rows = len(reviews) - reviews["order_id"].nunique()

print("Additional duplicated order_id rows in orders:", duplicated_order_rows)
print("Additional duplicated review_id rows:", duplicated_review_id_rows)
print("Completely identical review rows:", exact_duplicate_review_rows)
print("Orders with multiple review records:", orders_with_multiple_reviews)
print("Extra review rows beyond one row per order:", extra_review_rows)

Additional duplicated order_id rows in orders: 0
Additional duplicated review_id rows: 814
Completely identical review rows: 0
Orders with multiple review records: 547
Extra review rows beyond one row per order: 551


In [4]:
multiple_review_order_ids = review_counts_per_order[
    review_counts_per_order > 1
].index

multiple_reviews = reviews[
    reviews["order_id"].isin(multiple_review_order_ids)
].copy()

multiple_review_summary = (
    multiple_reviews
    .groupby("order_id")
    .agg(
        review_rows=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        unique_review_scores=("review_score", "nunique"),
        earliest_answer=("review_answer_timestamp", "min"),
        latest_answer=("review_answer_timestamp", "max")
    )
    .reset_index()
)

same_score_orders = (
    multiple_review_summary["unique_review_scores"] == 1
).sum()

different_score_orders = (
    multiple_review_summary["unique_review_scores"] > 1
).sum()

print("Multiple-review orders with the same score:", same_score_orders)
print("Multiple-review orders with different scores:", different_score_orders)

display(
    multiple_review_summary[
        multiple_review_summary["unique_review_scores"] > 1
    ].head(10)
)

Multiple-review orders with the same score: 345
Multiple-review orders with different scores: 202


,order_id,review_rows,unique_review_ids,unique_review_scores,earliest_answer,latest_answer
1,013056cfe49763c6f66bda03396c5ee3,2,2,2,2018-02-23 12:12:30,2018-03-05 17:02:00
3,02355020fd0a40a0d56df9f6ff060413,2,2,2,2018-03-22 01:32:08,2018-03-30 03:16:19
4,029863af4b968de1e5d6a82782e662f5,2,2,2,2017-07-17 13:58:06,2017-07-20 12:06:11
8,03c939fd7fd3b38f8485a0f95798f1f6,3,3,2,2018-03-06 19:50:32,2018-03-30 00:29:09
9,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,2,2018-02-23 23:55:53,2018-03-09 00:09:40
10,04f1827088d972a62224f5203a071500,2,2,2,2018-01-03 10:40:06,2018-01-07 11:14:04
11,0544030711e50ec2cb6c15764d22891a,2,2,2,2018-05-04 11:29:18,2018-05-04 17:54:45
14,059bbeb3477ed66fd7e670c3f879009a,2,2,2,2017-08-01 20:39:04,2017-08-04 09:32:59
17,073fa4be4665f397a289842b1053229c,2,2,2,2018-01-24 20:26:57,2018-01-25 10:01:21
19,075a544c5f4ed4bb75f82b160465fe76,2,2,2,2018-03-07 12:32:38,2018-03-17 19:39:26


## 3. Select One Review per Order

Some orders contain multiple review records, and 202 of these orders have different review scores.

To preserve one row per order, the main analysis retains the latest review according to `review_answer_timestamp`. This rule treats the customer's latest submitted response as the final evaluation.

Alternative review-selection rules may be examined in a sensitivity analysis.}

In [6]:
reviews_one_per_order = (
    reviews
    .sort_values(
        by=[
            "order_id",
            "review_answer_timestamp",
            "review_creation_date",
            "review_id"
        ],
        na_position="first"
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .copy()
)

removed_review_rows = len(reviews) - len(reviews_one_per_order)

print("Original review rows:", len(reviews))
print("Review rows after selection:", len(reviews_one_per_order))
print("Removed additional review rows:", removed_review_rows)
print(
    "Duplicated order_id rows after selection:",
    reviews_one_per_order["order_id"].duplicated().sum()
)

Original review rows: 99224
Review rows after selection: 98673
Removed additional review rows: 551
Duplicated order_id rows after selection: 0


## 4. Define Eligible Delivered Orders

The analysis is restricted to delivered orders with non-missing purchase, actual-delivery and estimated-delivery timestamps.

Approval and carrier timestamps are not used as mandatory exclusion criteria because they are not required to calculate the main delivery-delay measure.

Orders with an actual delivery date earlier than the purchase date are treated as logically invalid.

In [7]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

required_date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_with_required_dates = (
    delivered_orders
    .dropna(subset=required_date_columns)
    .copy()
)

orders_with_required_dates["delivery_days"] = (
    orders_with_required_dates["order_delivered_customer_date"].dt.normalize()
    - orders_with_required_dates["order_purchase_timestamp"].dt.normalize()
).dt.days

invalid_delivery_time_count = (
    orders_with_required_dates["delivery_days"] < 0
).sum()

eligible_orders = orders_with_required_dates[
    orders_with_required_dates["delivery_days"] >= 0
].copy()

print("Original orders:", len(orders))
print("Delivered orders:", len(delivered_orders))
print("Delivered orders with required dates:", len(orders_with_required_dates))
print("Orders delivered before purchase:", invalid_delivery_time_count)
print("Eligible delivered orders:", len(eligible_orders))
print(
    "Duplicated order_id rows:",
    eligible_orders["order_id"].duplicated().sum()
)

Original orders: 99441
Delivered orders: 96478
Delivered orders with required dates: 96470
Orders delivered before purchase: 0
Eligible delivered orders: 96470
Duplicated order_id rows: 0


## 5. Calculate Delivery-Delay Measures

`delay_days` is calculated as the actual delivery date minus the estimated delivery date.

- A positive value indicates a delayed order.
- Zero indicates delivery on the estimated date.
- A negative value indicates early delivery.

Early-delivery observations are retained because they represent valid delivery performance.

In [8]:
eligible_orders["delay_days"] = (
    eligible_orders["order_delivered_customer_date"].dt.normalize()
    - eligible_orders["order_estimated_delivery_date"].dt.normalize()
).dt.days

eligible_orders["is_delayed"] = (
    eligible_orders["delay_days"] > 0
).astype(int)

eligible_orders["delivery_status"] = (
    eligible_orders["is_delayed"]
    .map({
        0: "On time or early",
        1: "Delayed"
    })
)

delivery_status_summary = (
    eligible_orders["delivery_status"]
    .value_counts()
    .rename_axis("delivery_status")
    .reset_index(name="order_count")
)

delivery_status_summary["order_percent"] = (
    delivery_status_summary["order_count"]
    / len(eligible_orders)
    * 100
).round(2)

display(delivery_status_summary)

print("Minimum delay days:", eligible_orders["delay_days"].min())
print("Median delay days:", eligible_orders["delay_days"].median())
print("Maximum delay days:", eligible_orders["delay_days"].max())

,delivery_status,order_count,order_percent
0,On time or early,89936,93.23
1,Delayed,6534,6.77


Minimum delay days: -147
Median delay days: -12.0
Maximum delay days: 188


## 6. Merge Delivery Information with Customer Reviews

The eligible orders table is merged with the deduplicated reviews table using `order_id`.

An inner join retains only orders that have both valid delivery information and an available customer-review score. The merge is validated as one-to-one to prevent accidental duplication of orders.

In [9]:
review_columns = [
    "order_id",
    "review_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

core_dataset = eligible_orders.merge(
    reviews_one_per_order[review_columns],
    on="order_id",
    how="inner",
    validate="one_to_one"
)

orders_without_reviews = (
    len(eligible_orders) - len(core_dataset)
)

print("Eligible delivered orders:", len(eligible_orders))
print("Orders included after review merge:", len(core_dataset))
print("Eligible orders without reviews:", orders_without_reviews)
print(
    "Duplicated order_id rows in core dataset:",
    core_dataset["order_id"].duplicated().sum()
)
print(
    "Missing review scores:",
    core_dataset["review_score"].isna().sum()
)
print("Core dataset shape:", core_dataset.shape)

Eligible delivered orders: 96470
Orders included after review merge: 95824
Eligible orders without reviews: 646
Duplicated order_id rows in core dataset: 0
Missing review scores: 0
Core dataset shape: (95824, 16)


## 7. Validate the Core Dataset

The final dataset must contain one unique order per row, valid review scores, and complete delivery-delay measures.

The main relationship between delivery status and customer satisfaction is then recalculated using the validated order-level dataset.

In [10]:
assert core_dataset["order_id"].is_unique
assert core_dataset["review_score"].between(1, 5).all()
assert core_dataset["delay_days"].notna().all()
assert core_dataset["delivery_days"].ge(0).all()

core_dataset["low_rating"] = (
    core_dataset["review_score"] <= 2
).astype(int)

core_summary = (
    core_dataset
    .groupby("delivery_status")
    .agg(
        order_count=("order_id", "count"),
        average_review_score=("review_score", "mean"),
        median_review_score=("review_score", "median"),
        low_rating_rate=("low_rating", "mean")
    )
    .reset_index()
)

core_summary["order_percent"] = (
    core_summary["order_count"]
    / len(core_dataset)
    * 100
).round(2)

core_summary["average_review_score"] = (
    core_summary["average_review_score"].round(2)
)

core_summary["low_rating_percent"] = (
    core_summary["low_rating_rate"] * 100
).round(2)

core_summary = core_summary[
    [
        "delivery_status",
        "order_count",
        "order_percent",
        "average_review_score",
        "median_review_score",
        "low_rating_percent"
    ]
]

display(core_summary)

print("All validation checks passed.")

,delivery_status,order_count,order_percent,average_review_score,median_review_score,low_rating_percent
0,Delayed,6381,6.66,2.27,1.0,62.42
1,On time or early,89443,93.34,4.29,5.0,9.27


All validation checks passed.


## Interim Findings

The validated core dataset contains 95,824 unique delivered orders with customer reviews.

Delayed orders account for 6.66% of the reviewed sample. Their average review score is 2.27, compared with 4.29 for orders delivered on time or early.

The median review score is 1 for delayed orders and 5 for on-time or early orders. In addition, 62.42% of delayed orders receive a low rating of 1 or 2, compared with only 9.27% of on-time or early orders.

These results indicate a strong association between delivery delays and lower customer satisfaction. They should not yet be interpreted as definitive causal effects.

In [11]:
processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(
    parents=True,
    exist_ok=True
)

core_dataset_path = (
    processed_data_dir
    / "core_analysis_orders.csv"
)

core_dataset.to_csv(
    core_dataset_path,
    index=False,
    encoding="utf-8-sig"
)

print("Exported file:", core_dataset_path.resolve())
print("Exported rows:", len(core_dataset))
print("Exported columns:", len(core_dataset.columns))
print(
    "File size (MB):",
    round(core_dataset_path.stat().st_size / (1024 ** 2), 2)
)

Exported file: F:\Personal_interesting_projects\ecommerce-delivery-analysis\Main file\ecommerce-delivery-analysis\data\processed\core_analysis_orders.csv
Exported rows: 95824
Exported columns: 17
File size (MB): 24.63


In [12]:
export_check = pd.read_csv(
    core_dataset_path
)

assert export_check.shape == core_dataset.shape
assert export_check["order_id"].is_unique
assert export_check["review_score"].between(1, 5).all()
assert export_check["delay_days"].notna().all()

print("Exported CSV validation passed.")
print("Verified shape:", export_check.shape)

Exported CSV validation passed.
Verified shape: (95824, 17)


In [13]:
gitignore_path = project_root / ".gitignore"
ignore_rule = "data/processed/"

current_gitignore = gitignore_path.read_text(
    encoding="utf-8",
    errors="replace"
)

if ignore_rule not in current_gitignore.splitlines():
    with gitignore_path.open(
        mode="a",
        encoding="utf-8"
    ) as file:
        if not current_gitignore.endswith("\n"):
            file.write("\n")
        file.write("\n# Locally generated processed data\n")
        file.write("data/processed/\n")

    print("Added to .gitignore:", ignore_rule)
else:
    print("Rule already exists:", ignore_rule)

print("Gitignore file exists:", gitignore_path.exists())

Added to .gitignore: data/processed/
Gitignore file exists: True
